In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
data = np.load("./Netflix Data.npz", allow_pickle=True)

P = data["P"]
Q = data["Q"]
bu = data["bu"]
bm = data["bm"]
mu = data["mu"]
movie_titles = data["movie_titles"]
movie_years = data["movie_years"]

In [4]:
sorted_index = np.argsort(bm)
last_ten = sorted_index[-10:]
top_ten_movies = bm[last_ten]

In [ ]:
print(top_ten_movies)
print("-"*10)

# Finding top Ten Movies
for i in last_ten:
    print(f"Movie Name: {movie_titles[i]}\nMovie Year: {movie_years[i]}\nMovie Bias: {bm[i]}")

[0.97289828 1.01716329 1.02090649 1.02096696 1.03946621 1.06878517
 1.10243022 1.11454703 1.11813094 1.12168797]
----------
Movie Name: The Sopranos: Season 5
Movie Year: 2004
Movie Bias: 0.9728982831969061
----------
Movie Name: House, M.D.: Season 1
Movie Year: 2004
Movie Bias: 1.0171632923433136
----------
Movie Name: The Shawshank Redemption: Special Edition
Movie Year: 1994
Movie Bias: 1.0209064898782854
----------
Movie Name: Anne of Green Gables: The Sequel
Movie Year: 1987
Movie Bias: 1.0209669607009813
----------
Movie Name: Veronica Mars: Season 1
Movie Year: 2004
Movie Bias: 1.039466210798041
----------
Movie Name: Battlestar Galactica: Season 1
Movie Year: 2004
Movie Bias: 1.0687851663980024
----------
Movie Name: Lord of the Rings: The Two Towers: Extended Edition
Movie Year: 2002
Movie Bias: 1.1024302171919322
----------
Movie Name: Lost: Season 1
Movie Year: 2004
Movie Bias: 1.1145470332414722
----------
Movie Name: The Lord of the Rings: The Fellowship of the Ring: Exte

In [6]:
def search_movie(query, movie_titles, movie_years):
    """
    Given a query string like "godfather", find all movies 
    whose title contains that string (case-insensitive).
    
    Return a list of tuples: (index, title, year)
    """
    query = query.lower()
    for index, movie in enumerate(movie_titles):
        movie = movie.lower()
        if query in movie:
            year = movie_years[index]
            return (index, movie, year)
    return (-1, "No Movie Found")

In [7]:
ans = search_movie("The Sopranos: Season 5", movie_titles=movie_titles, movie_years=movie_years)

print(ans)
# print(Q.shape)
# print(Q[0])

(11661, 'the sopranos: season 5', '2004')


In [8]:
def find_similar_movies(movie_index, Q, movie_titles, movie_years, top_n=10):
    """
    Given a movie_index, find the top_n most similar movies.
    
    Steps:
    1. Get the target movie vector: Q[movie_index]
    2. Compute cosine similarity with every other movie
    
       cosine_similarity = dot(A, B) / (norm(A) * norm(B))

    3. Sort by similarity (highest first)
    4. Return top_n results (skip index 0 since that's the movie itself)
    """
    target_vector = Q[movie_index]
    cosine_sim = cosine_similarity([target_vector], Q).flatten()
    sorted_array = np.argsort(cosine_sim)
    top_indices = sorted_array[-(top_n + 1):-1]
    top_indices = top_indices[::-1]

    return top_indices[1:11]

In [9]:
# cosine_sim = cosine_similarity([Q[0]], Q)
# sorted_array = np.sort(cosine_sim)

# print(sorted_array)

In [10]:
ans = find_similar_movies(movie_index=11661, Q=Q, movie_titles=movie_titles, movie_years=movie_years, top_n=10)
print(ans)

for index in ans:
    print(movie_titles[index])

[14301  5759  8115 16005 10642 10777 15295  3289  9817]
The Sopranos: Season 2
The Sopranos: Season 3
The Sopranos: Season 4
Seinfeld: Seasons 1 & 2
Seinfeld: Season 4
Scrubs: Season 1
Band of Brothers
The Godfather, Part II
L.A. Confidential


In [11]:
# Search for a movie
results = search_movie(query="House, M.D.: Season 1", movie_titles=movie_titles, movie_years=movie_years)
# for idx, title, year in results:
#     print(f"[{idx}] {title} ({year})")
print(results)

# Pick one and find similar movies
# Use the index from the search results
# 
similar = find_similar_movies(movie_index=results[0], Q=Q, movie_titles=movie_titles, movie_years=movie_years, top_n=10)

print(similar)

for index in similar:
    print(movie_titles[index])

(13503, 'house, m.d.: season 1', '2004')
[12955 14470 15182 14319 15593  7030 15973 12197 16106]
Angel: Season 2
Santitos
MASH: Season 4
MASH: Season 2
MASH: Season 1
Support Your Local Sheriff!
Yankee Doodle Dandy
100 Years of the New York Yankees
MASH: Season 3


---

# SQL

In [12]:
import sqlite3
from pydantic import BaseModel

In [13]:
DB_NAME = "cart.db"

def init_db1():
    connection_obj = sqlite3.connect(DB_NAME)
    cursor_obj = connection_obj.cursor()
    table_creation_query = """
    CREATE TABLE IF NOT EXISTS cart (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id TEXT NOT NULL,
        movie_index INTEGER NOT NULL,
        movie_name TEXT NOT NULL,
        movie_year INTEGER NOT NULL,
        added_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE(session_id, movie_index)
    );
    """
    cursor_obj.execute(table_creation_query)
    connection_obj.commit()
    connection_obj.close()
    print("Table is ready")

def add_to_cart1(session_id: str, movie_index: int, movie_name: str, movie_year: int):
    connection_obj = sqlite3.connect(DB_NAME)
    cursor_obj = connection_obj.cursor()
    query = """
    INSERT INTO cart (
        session_id,
        movie_index,
        movie_name,
        movie_year
    ) VALUES (?, ?, ?, ?);
    """
    data = (session_id, movie_index, movie_name, movie_year)
    cursor_obj.execute(query, data)
    connection_obj.commit()
    print(f"{movie_name} added successfully")
    connection_obj.close()

def get_cart1(session_id: str):
    """
    Return all cart items for a given session_id.
    
    Hints:
    - cursor.execute("SELECT ... WHERE session_id = ?", (session_id,))
    - cursor.fetchall() returns list of tuples
    """
    connection_obj = sqlite3.connect(DB_NAME)
    cursor_obj = connection_obj.cursor()
    query = """
    SELECT * FROM cart WHERE session_id = ?;
    """
    cursor_obj.execute(query, (session_id,))
    rows = cursor_obj.fetchall()
    print(f"{session_id} session id returened")
    connection_obj.close()
    return rows


def clear_cart1(session_id: str):
    """
    Remove all items for a given session_id (checkout).
    
    Hints:
    - DELETE FROM ... WHERE session_id = ?
    """
    connection_obj = sqlite3.connect(DB_NAME)
    cursor_obj = connection_obj.cursor()
    query = """
    DELETE FROM cart WHERE session_id = ?;
    """
    cursor_obj.execute(query, (session_id,))
    connection_obj.commit()
    print(f"Checked out successfully!")
    connection_obj.close()

In [14]:
if __name__ == "__main__":
    init_db1()

    add_to_cart1("user123", 42, "The Godfather", "1972")
    add_to_cart1("user123", 99, "Pulp Fiction", "1994")
    add_to_cart1("user456", 10, "Finding Nemo", "2003")

    print("User123 cart:")
    print(get_cart1("user123"))

    print("\nUser456 cart:")
    print(get_cart1("user456"))

    clear_cart1("user123")
    print("\nUser123 cart after checkout:")
    print(get_cart1("user123"))

    print("\nUser456 cart (should be unchanged):")
    print(get_cart1("user456"))

Table is ready
The Godfather added successfully
Pulp Fiction added successfully
Finding Nemo added successfully
User123 cart:
user123 session id returened
[(1, 'user123', 42, 'The Godfather', 1972, '2026-03-13 07:08:20'), (2, 'user123', 99, 'Pulp Fiction', 1994, '2026-03-13 07:08:20')]

User456 cart:
user456 session id returened
[(3, 'user456', 10, 'Finding Nemo', 2003, '2026-03-13 07:08:20')]
Checked out successfully!

User123 cart after checkout:
user123 session id returened
[]

User456 cart (should be unchanged):
user456 session id returened
[(3, 'user456', 10, 'Finding Nemo', 2003, '2026-03-13 07:08:20')]


---

# Langchain Tools

In [15]:
!pip install langchain-ollama langchain-core

In [16]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

@tool
def test_tool():
    """Execute this tool when the user explicitly asks to run a system test or check connectivity."""
    return "Working"

llm = ChatOllama(model= "ministral-3:8b", temperature = 0)
llm_with_tools = llm.bind_tools([test_tool])

response = llm_with_tools.invoke("Use the test tool to verify the system status.")
print(response)
print(response.tool_calls)

content='' additional_kwargs={} response_metadata={'model': 'ministral-3:8b', 'created_at': '2026-03-13T07:08:31.76042Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8624031584, 'load_duration': 135454459, 'prompt_eval_count': 610, 'prompt_eval_duration': 7866835375, 'eval_count': 7, 'eval_duration': 601495000, 'logprobs': None, 'model_name': 'ministral-3:8b', 'model_provider': 'ollama'} id='lc_run--019ce606-5edf-79d0-89dc-5d229d441bfc-0' tool_calls=[{'name': 'test_tool', 'args': {}, 'id': '1c0d2ee4-2367-448b-8619-32ef702ccba7', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 610, 'output_tokens': 7, 'total_tokens': 617}
[{'name': 'test_tool', 'args': {}, 'id': '1c0d2ee4-2367-448b-8619-32ef702ccba7', 'type': 'tool_call'}]


In [17]:
def top_movies(bm, movie_titles, movie_years, n):
    sorted_index = np.argsort(bm)
    last_n = sorted_index[-n:]
    lines = []
    for i in last_n:
        lines.append(f"\nMovie Name: {movie_titles[i]}\nMovie Year: {movie_years[i]}")

    return "\n".join(lines)

ans = top_movies(bm=bm, movie_titles=movie_titles, movie_years=movie_years, n=10)
# print(ans)

In [18]:
n = 10
def find_similar_movies(movie_index, Q, top_n=n):
    target_vector = Q[movie_index]
    cosine_sim = cosine_similarity([target_vector], Q).flatten()
    sorted_array = np.argsort(cosine_sim)[::-1]
    top_indices = sorted_array[1:top_n + 1]
    result_list = []
    for index in range(len(top_indices)):
        result_list.append(movie_titles[index])
    return "\n".join(result_list)

print(movie_titles[100])
print(find_similar_movies(100, Q=Q))

Complete Shamanic Princess
Dinosaur Planet
Isle of Man TT 2004 Review
Character
Paula Abdul's Get Up & Dance
The Rise and Fall of ECW
Sick
8 Man
What the #$*! Do We Know!?
Class of Nuke 'Em High 2
Fighter


In [19]:
def get_cart(session_id: str):
    connection_obj = sqlite3.connect(DB_NAME)
    cursor_obj = connection_obj.cursor()
    query = """
    SELECT * FROM cart WHERE session_id = ?;
    """
    cursor_obj.execute(query, (session_id,))
    rows = cursor_obj.fetchall()
    # print(f"{session_id} session id returened")
    connection_obj.close()
    return rows

ans = get_cart(session_id="user456")
# 0 = id,
# 1 = session_id, 
# 2 = movie_index,
# 3 = movie_name, 
# 4 = movie_year, 
# 5 = added_at
print(ans)
print(len(ans))
print(f"id: {ans[0][0]}")
print(f"session_id: {ans[0][1]}")
print(f"movie_index: {ans[0][2]}")
print(f"movie_name: {ans[0][3]}")
print(f"movie_year: {ans[0][4]}")
print(f"added_at: {ans[0][5]}")

[(3, 'user456', 10, 'Finding Nemo', 2003, '2026-03-13 07:08:20')]
1
id: 3
session_id: user456
movie_index: 10
movie_name: Finding Nemo
movie_year: 2003
added_at: 2026-03-13 07:08:20
